In [1]:
# Ensure that dnq directory is updated to the latest version of code on GitHub
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
%cd drive/MyDrive/Robotics/grokking-deep-rl-ide-version

/content/drive/MyDrive/Robotics/grokking-deep-rl-ide-version


In [2]:
# %cd grokking-deep-rl-ide-version
%cd dnq_ddnq_chp_9

/content/drive/MyDrive/Robotics/grokking-deep-rl-ide-version/dnq_ddnq_chp_9


In [3]:
!git pull origin main

From https://github.com/drenfr01/grokking-deep-rl-ide-version
 * branch            main       -> FETCH_HEAD
Already up to date.


In [4]:
import warnings ; warnings.filterwarnings('ignore')
import os, sys

try:
    _nb_dir = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    _nb_dir = os.getcwd()
if _nb_dir not in sys.path:
    sys.path.insert(0, _nb_dir)

import time
import torch
import torch.optim as optim

import numpy as np
from IPython.display import display
import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
from textwrap import wrap

import os.path
import os

LEAVE_PRINT_EVERY_N_SECS = 60
ERASE_LINE = '\x1b[2K'
EPS = 1e-6
BEEP = lambda: os.system("printf '\a'")
RESULTS_DIR = os.path.join('..', 'results')
SEEDS = (12, 34, 56, 78, 90)

%matplotlib inline

In [5]:
plt.style.use('fivethirtyeight')
params = {
    'figure.figsize': (15, 8),
    'font.size': 24,
    'legend.fontsize': 20,
    'axes.titlesize': 28,
    'axes.labelsize': 24,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20
}
pylab.rcParams.update(params)
np.set_printoptions(suppress=True)

In [8]:
torch.cuda.is_available()

False

In [7]:
from exploration_strategies import EGreedyStrategy, EGreedyLinearStrategy, EGreedyExpStrategy, SoftMaxStrategy, GreedyStrategy
from networks import FCQ
from buffers import ReplayBuffer
from dnq import DQN
from utils import get_make_env_fn


In [33]:
dqn_results = []
best_agent, best_eval_score = None, float('-inf')
for seed in SEEDS:
    environment_settings = {
        'env_name': 'CartPole-v1',
        'gamma': 1.00,
        'max_minutes': 20,
        'max_episodes': 10000,
        'goal_mean_100_reward': 475
    }
    
    value_model_fn = lambda nS, nA: FCQ(nS, nA, hidden_dims=(512,128))
    value_optimizer_fn = lambda net, lr: optim.RMSprop(net.parameters(), lr=lr)
    value_optimizer_lr = 0.0005

    # training_strategy_fn = lambda: EGreedyStrategy(epsilon=0.5)
    # training_strategy_fn = lambda: EGreedyLinearStrategy(init_epsilon=1.0,
    #                                                      min_epsilon=0.3, 
    #                                                      max_steps=20000)
    # training_strategy_fn = lambda: SoftMaxStrategy(init_temp=1.0, 
    #                                                min_temp=0.1, 
    #                                                exploration_ratio=0.8, 
    #                                                max_steps=20000)
    training_strategy_fn = lambda: EGreedyExpStrategy(init_epsilon=1.0,  
                                                      min_epsilon=0.3, 
                                                      decay_steps=20000)
    evaluation_strategy_fn = lambda: GreedyStrategy()

    replay_buffer_fn = lambda: ReplayBuffer(max_size=50000, batch_size=64)
    n_warmup_batches = 5
    update_target_every_steps = 10

    env_name, gamma, max_minutes, \
    max_episodes, goal_mean_100_reward = environment_settings.values()
    agent = DQN(replay_buffer_fn,
                value_model_fn,
                value_optimizer_fn,
                value_optimizer_lr,
                training_strategy_fn,
                evaluation_strategy_fn,
                n_warmup_batches,
                update_target_every_steps)

    make_env_fn, make_env_kargs = get_make_env_fn(env_name=env_name)
    result, final_eval_score, training_time, wallclock_time = agent.train(
        make_env_fn, make_env_kargs, seed, gamma, max_minutes, max_episodes, goal_mean_100_reward)
    dqn_results.append(result)
    if final_eval_score > best_eval_score:
        best_eval_score = final_eval_score
        best_agent = agent
dqn_results = np.array(dqn_results)
_ = BEEP()

AttributeError: 'TimeLimit' object has no attribute 'seed'